# Stage 3.3 — Phase A
# Train Feature Engineering Implementation & Sanity Check

## Objective and frozen scope

This Notebook independently reloads the formal raw CSV, exactly replays the locked Stage 3.1 split, verifies both ID-set hashes, seals Holdout Test, and applies only the frozen Feature Engineering rules to Train.

Phase A creates exactly two cleaned categorical fields and six approved new features. It does not transform Test, reassess feature usefulness, compare anything with target, perform WOE/binning, handle class imbalance, train a model, or begin Phase B.

## Test-sealing contract

Test exists only long enough to verify row count, ID separation/coverage, and the locked stable hash. After those assertions it is deleted from the live state. Every mapping, calculation, sample, and descriptive output below receives Train only.


In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

TARGET = "default.payment.next.month"
ID_COL = "ID"
RANDOM_STATE = 42
TEST_SIZE = 0.20
EXPECTED_TRAIN_HASH = "bcb37af63ae7005df78e6140837594b5d650d27148b46b5d51354430ad958857"
EXPECTED_TEST_HASH = "d6748249f9f9fb2855746e0ed980b5013047267104090a14aa8138a700fe6f0f"

PAY_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_COLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAY_AMT_COLS = [f"PAY_AMT{i}" for i in range(1, 7)]
GROUP_FIELDS = ["EDUCATION_GROUP", "MARRIAGE_GROUP"]
NEW_FEATURES = [
    "delinquent_months", "max_delinquency", "severe_delinquent_months",
    "current_utilization", "avg_utilization", "max_utilization",
]

PREDICTOR_COLUMNS = [
    "LIMIT_BAL", "SEX", "EDUCATION_GROUP", "MARRIAGE_GROUP", "AGE",
    *PAY_COLS, *BILL_COLS, *PAY_AMT_COLS, *NEW_FEATURES,
]
RAW_PRESERVED_COLUMNS = [
    ID_COL, TARGET, "AGE", "SEX", "LIMIT_BAL", "EDUCATION", "MARRIAGE",
    *PAY_COLS, *BILL_COLS, *PAY_AMT_COLS,
]

assert len(PREDICTOR_COLUMNS) == 29
assert len(set(PREDICTOR_COLUMNS)) == 29

def locate_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data/raw/UCI_Credit_Card.csv").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the formal raw CSV from the launch directory or its parents.")

def stable_id_set_hash(values):
    canonical = "\n".join(str(int(value)) for value in sorted(values))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

project_root = locate_project_root()
data_path = project_root / "data/raw/UCI_Credit_Card.csv"
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Project root:", project_root)
print("Raw CSV:", data_path)
print("Frozen predictor count:", len(PREDICTOR_COLUMNS))


Python executable: /Users/yu/2.工作/Risk Analysis Project/.venv/bin/python
Python version: 3.9.6
Project root: /Users/yu/2.工作/Risk Analysis Project
Raw CSV: /Users/yu/2.工作/Risk Analysis Project/data/raw/UCI_Credit_Card.csv
Frozen predictor count: 29


## 1. Load the formal raw CSV and confirm structure

This is a fresh load; no prior Notebook state is reused. Only structural checks occur before the locked split.


In [2]:
raw_df = pd.read_csv(data_path)
assert raw_df.shape == (30000, 25)
assert raw_df[ID_COL].is_unique
assert raw_df.isna().sum().sum() == 0
assert set(raw_df[TARGET].unique()) == {0, 1}
assert np.isfinite(raw_df.select_dtypes(include=np.number).to_numpy()).all()

raw_confirmation = pd.DataFrame({
    "value": [
        raw_df.shape[0], raw_df.shape[1], raw_df[ID_COL].nunique(),
        int(raw_df.isna().sum().sum()), int(raw_df.duplicated().sum()),
        sorted(raw_df[TARGET].unique().tolist()), data_path.stat().st_size,
    ]
}, index=[
    "rows", "columns", "unique_IDs", "missing_cells", "duplicate_rows",
    "target_value_set_only", "raw_file_bytes",
])
display(raw_confirmation)


,value
rows,30000
columns,25
unique_IDs,30000
missing_cells,0
duplicate_rows,0
target_value_set_only,"[0, 1]"
raw_file_bytes,2862995


## 2. Replay the locked Stage 3.1 split and seal Test

Canonical ID-set hashing: cast IDs to canonical decimal integer strings, sort numerically, join with one newline, add no trailing newline, UTF-8 encode, then SHA-256.

Feature Engineering cannot start unless both hashes exactly match Stage 3.1.


In [3]:
train_raw, sealed_test = train_test_split(
    raw_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=raw_df[TARGET],
)

train_ids = set(train_raw[ID_COL].astype(int))
test_ids = set(sealed_test[ID_COL].astype(int))
full_ids = set(raw_df[ID_COL].astype(int))
train_id_hash = stable_id_set_hash(train_ids)
test_id_hash = stable_id_set_hash(test_ids)

assert len(raw_df) == 30000
assert len(train_raw) == 24000
assert len(sealed_test) == 6000
assert train_ids.isdisjoint(test_ids)
assert train_ids | test_ids == full_ids
assert len(train_ids) + len(test_ids) == len(full_ids)
assert train_id_hash == EXPECTED_TRAIN_HASH
assert test_id_hash == EXPECTED_TEST_HASH

split_evidence = pd.DataFrame({
    "value": [
        "train_test_split", "80% / 20%", RANDOM_STATE, TARGET,
        len(train_raw), len(sealed_test), len(train_ids & test_ids),
        train_ids | test_ids == full_ids, train_id_hash, test_id_hash,
        "numeric sort → decimal strings → newline join → no trailing newline → UTF-8 → SHA-256",
    ]
}, index=[
    "method", "ratio", "random_state", "stratification_target", "train_rows",
    "test_rows", "ID_overlap", "full_ID_coverage", "train_ID_SHA256",
    "test_ID_SHA256", "canonicalization",
])
display(split_evidence)
print("Train ID-set SHA-256:", train_id_hash)
print("Test ID-set SHA-256:", test_id_hash)

# Preserve the immutable Train audit baseline, then remove Test and full raw data.
train_original = train_raw.copy(deep=True)
del sealed_test, raw_df, test_ids, full_ids
TEST_ISOLATION_STATUS = "SEALED — Test deleted after row/ID/hash assertions; no Phase A transformation applied"
print(TEST_ISOLATION_STATUS)


,value
method,train_test_split
ratio,80% / 20%
random_state,42
stratification_target,default.payment.next.month
train_rows,24000
test_rows,6000
ID_overlap,0
full_ID_coverage,True
train_ID_SHA256,bcb37af63ae7005df78e6140837594b5d650d27148b46b...
test_ID_SHA256,d6748249f9f9fb2855746e0ed980b5013047267104090a...


Train ID-set SHA-256: bcb37af63ae7005df78e6140837594b5d650d27148b46b5d51354430ad958857
Test ID-set SHA-256: d6748249f9f9fb2855746e0ed980b5013047267104090a14aa8138a700fe6f0f
SEALED — Test deleted after row/ID/hash assertions; no Phase A transformation applied


## 3. Train-only cleaned category fields

These are frozen modeling groupings, not claims that undocumented source codes have identical business meaning. EDUCATION code 4 is documented as Others; exact meanings of 0/5/6 remain unconfirmed. MARRIAGE code 0 also remains unconfirmed.

Raw EDUCATION and MARRIAGE remain in the Train audit frame for reconciliation, but only the grouped fields enter the predictor-ready view.


In [4]:
EDUCATION_MAP = {
    1: "Graduate School", 2: "University", 3: "High School",
    0: "Other / Unknown", 4: "Other / Unknown",
    5: "Other / Unknown", 6: "Other / Unknown",
}
MARRIAGE_MAP = {
    1: "Married", 2: "Single", 0: "Other / Unknown", 3: "Other / Unknown",
}
EXPECTED_EDUCATION_GROUPS = {"Graduate School", "University", "High School", "Other / Unknown"}
EXPECTED_MARRIAGE_GROUPS = {"Married", "Single", "Other / Unknown"}

train_work = train_raw.copy(deep=True)
train_work["EDUCATION_GROUP"] = train_work["EDUCATION"].map(EDUCATION_MAP)
train_work["MARRIAGE_GROUP"] = train_work["MARRIAGE"].map(MARRIAGE_MAP)

assert train_work["EDUCATION_GROUP"].notna().all()
assert train_work["MARRIAGE_GROUP"].notna().all()
assert set(train_work["EDUCATION_GROUP"].unique()) == EXPECTED_EDUCATION_GROUPS
assert set(train_work["MARRIAGE_GROUP"].unique()) == EXPECTED_MARRIAGE_GROUPS

education_reconciliation = (
    train_work.groupby(["EDUCATION", "EDUCATION_GROUP"], observed=True)
    .size().rename("Train_count").reset_index().sort_values("EDUCATION")
)
marriage_reconciliation = (
    train_work.groupby(["MARRIAGE", "MARRIAGE_GROUP"], observed=True)
    .size().rename("Train_count").reset_index().sort_values("MARRIAGE")
)

education_group_counts = train_work["EDUCATION_GROUP"].value_counts().sort_index()
marriage_group_counts = train_work["MARRIAGE_GROUP"].value_counts().sort_index()
assert education_reconciliation["Train_count"].sum() == len(train_work) == education_group_counts.sum()
assert marriage_reconciliation["Train_count"].sum() == len(train_work) == marriage_group_counts.sum()
assert education_group_counts.equals(train_work["EDUCATION"].map(EDUCATION_MAP).value_counts().sort_index())
assert marriage_group_counts.equals(train_work["MARRIAGE"].map(MARRIAGE_MAP).value_counts().sort_index())

print("EDUCATION raw-code reconciliation:")
display(education_reconciliation)
print("EDUCATION grouped counts:")
display(education_group_counts.rename("Train_count").to_frame())
print("MARRIAGE raw-code reconciliation:")
display(marriage_reconciliation)
print("MARRIAGE grouped counts:")
display(marriage_group_counts.rename("Train_count").to_frame())


EDUCATION raw-code reconciliation:


,EDUCATION,EDUCATION_GROUP,Train_count
0,0,Other / Unknown,12
1,1,Graduate School,8455
2,2,University,11256
3,3,High School,3903
4,4,Other / Unknown,97
5,5,Other / Unknown,235
6,6,Other / Unknown,42


EDUCATION grouped counts:


,Train_count
EDUCATION_GROUP,
Graduate School,8455
High School,3903
Other / Unknown,386
University,11256


MARRIAGE raw-code reconciliation:


,MARRIAGE,MARRIAGE_GROUP,Train_count
0,0,Other / Unknown,47
1,1,Married,10892
2,2,Single,12806
3,3,Other / Unknown,255


MARRIAGE grouped counts:


,Train_count
MARRIAGE_GROUP,
Married,10892
Other / Unknown,302
Single,12806


## 4. Train-only delinquency summaries

- `delinquent_months`: number of six raw PAY values greater than 0.
- `max_delinquency`: maximum after flooring PAY at 0 inside this calculation only.
- `severe_delinquent_months`: number of six raw PAY values greater than or equal to 2.

Raw PAY fields are not mapped, overwritten, binned, or assigned new business meanings.


In [5]:
pay_matrix = train_work[PAY_COLS]
train_work["delinquent_months"] = (pay_matrix > 0).sum(axis=1).astype("int64")
train_work["max_delinquency"] = pay_matrix.clip(lower=0).max(axis=1).astype("int64")
train_work["severe_delinquent_months"] = (pay_matrix >= 2).sum(axis=1).astype("int64")

print("Created delinquency features:", NEW_FEATURES[:3])


Created delinquency features: ['delinquent_months', 'max_delinquency', 'severe_delinquent_months']


## 5. Train-only utilization features

Within the calculation only, each BILL_AMT is floored at zero. Raw BILL fields remain untouched. Utilization is not capped at 1, clipped, winsorized, or transformed.


In [6]:
assert (train_work["LIMIT_BAL"] > 0).all()
positive_bill_matrix = train_work[BILL_COLS].clip(lower=0)
train_work["current_utilization"] = positive_bill_matrix["BILL_AMT1"] / train_work["LIMIT_BAL"]
train_work["avg_utilization"] = positive_bill_matrix.mean(axis=1) / train_work["LIMIT_BAL"]
train_work["max_utilization"] = positive_bill_matrix.max(axis=1) / train_work["LIMIT_BAL"]
del positive_bill_matrix

print("Created utilization features:", NEW_FEATURES[3:])
print("No positive_bill intermediate columns were saved.")


Created utilization features: ['current_utilization', 'avg_utilization', 'max_utilization']
No positive_bill intermediate columns were saved.


## 6. Predictor-ready Train structure

The 29-column view contains the frozen predictors only. ID and target are retained separately for traceability/labels. Raw EDUCATION and MARRIAGE remain audit fields but are intentionally excluded from predictors to avoid duplicate expression.


In [7]:
X_train = train_work[PREDICTOR_COLUMNS].copy()
train_trace = train_work[[ID_COL, TARGET]].copy()

assert X_train.columns.tolist() == PREDICTOR_COLUMNS
assert X_train.columns.is_unique
assert X_train.shape == (24000, 29)
assert ID_COL not in X_train.columns
assert TARGET not in X_train.columns
assert "EDUCATION" not in X_train.columns
assert "MARRIAGE" not in X_train.columns
assert train_trace.shape == (24000, 2)

schema_table = pd.DataFrame({
    "position": range(1, len(PREDICTOR_COLUMNS) + 1),
    "predictor": PREDICTOR_COLUMNS,
    "dtype": [str(X_train[c].dtype) for c in PREDICTOR_COLUMNS],
})
display(schema_table)
print("Predictor count:", X_train.shape[1])
print("Traceability/label fields:", train_trace.columns.tolist())


,position,predictor,dtype
0,1,LIMIT_BAL,float64
1,2,SEX,int64
2,3,EDUCATION_GROUP,object
3,4,MARRIAGE_GROUP,object
4,5,AGE,int64
5,6,PAY_0,int64
6,7,PAY_2,int64
7,8,PAY_3,int64
8,9,PAY_4,int64
9,10,PAY_5,int64


Predictor count: 29
Traceability/label fields: ['ID', 'default.payment.next.month']


## 7. Structural and raw-column integrity assertions

Every original Train row and preserved raw field is compared with the pre-transformation deep copy. This verifies that only the two grouped fields and six approved features were added.


In [8]:
pd.testing.assert_frame_equal(
    train_work[RAW_PRESERVED_COLUMNS],
    train_original[RAW_PRESERVED_COLUMNS],
    check_dtype=True,
    check_exact=True,
)
assert len(train_work) == len(train_original) == 24000
assert train_work.index.equals(train_original.index)
assert train_work[ID_COL].equals(train_original[ID_COL])
assert train_work[TARGET].equals(train_original[TARGET])
assert train_work[ID_COL].is_unique
assert set(train_work[ID_COL].astype(int)) == train_ids
assert train_work.duplicated(subset=[ID_COL]).sum() == 0
assert set(train_work.columns) - set(train_original.columns) == set(GROUP_FIELDS + NEW_FEATURES)
assert train_original.columns.tolist() == train_raw.columns.tolist()

numeric_predictors = X_train.select_dtypes(include=np.number)
assert X_train[GROUP_FIELDS].notna().all().all()
assert X_train.isna().sum().sum() == 0
assert np.isfinite(numeric_predictors.to_numpy()).all()

structural_checks = pd.DataFrame({
    "result": [
        "PASS — 24,000 rows preserved", "PASS — row order/index preserved",
        "PASS — ID set/order preserved and unique", "PASS — target unchanged",
        "PASS — AGE/SEX/LIMIT_BAL unchanged", "PASS — six raw PAY unchanged",
        "PASS — six raw BILL_AMT unchanged", "PASS — six raw PAY_AMT unchanged",
        "PASS — exactly 2 grouped fields + 6 approved features added",
        "PASS — 29 unique predictors exactly match frozen schema",
        "PASS — no NaN or numeric infinity",
    ]
}, index=[
    "row_count", "row_alignment", "ID_integrity", "target_integrity",
    "retained_demographic_credit", "raw_PAY_integrity", "raw_BILL_integrity",
    "raw_PAY_AMT_integrity", "added_columns", "predictor_schema", "finite_complete_values",
])
display(structural_checks)


,result
row_count,"PASS — 24,000 rows preserved"
row_alignment,PASS — row order/index preserved
ID_integrity,PASS — ID set/order preserved and unique
target_integrity,PASS — target unchanged
retained_demographic_credit,PASS — AGE/SEX/LIMIT_BAL unchanged
raw_PAY_integrity,PASS — six raw PAY unchanged
raw_BILL_integrity,PASS — six raw BILL_AMT unchanged
raw_PAY_AMT_integrity,PASS — six raw PAY_AMT unchanged
added_columns,PASS — exactly 2 grouped fields + 6 approved f...
predictor_schema,PASS — 29 unique predictors exactly match froz...


## 8. Delinquency formula sanity checks

The three columns are independently recomputed and compared row by row before range/logic assertions.


In [9]:
recalc_delinquent = (train_original[PAY_COLS] > 0).sum(axis=1).astype("int64")
recalc_max = train_original[PAY_COLS].apply(lambda col: col.clip(lower=0)).max(axis=1).astype("int64")
recalc_severe = (train_original[PAY_COLS] >= 2).sum(axis=1).astype("int64")
max_raw_positive_pay = int(max(0, train_original[PAY_COLS].to_numpy().max()))

assert train_work["delinquent_months"].equals(recalc_delinquent)
assert train_work["max_delinquency"].equals(recalc_max)
assert train_work["severe_delinquent_months"].equals(recalc_severe)
assert train_work["delinquent_months"].between(0, 6).all()
assert train_work["severe_delinquent_months"].between(0, 6).all()
assert (train_work["severe_delinquent_months"] <= train_work["delinquent_months"]).all()
assert (train_work["max_delinquency"] >= 0).all()
assert (train_work["max_delinquency"] <= max_raw_positive_pay).all()
assert (train_work.loc[train_work["delinquent_months"] == 0, "max_delinquency"] == 0).all()
assert (train_work.loc[train_work["severe_delinquent_months"] > 0, "max_delinquency"] >= 2).all()

delinquency_checks = pd.DataFrame({
    "result": [
        "PASS", "PASS", "PASS", "PASS", "PASS", "PASS", "PASS",
        f"PASS — maximum observed raw positive PAY code = {max_raw_positive_pay}",
    ]
}, index=[
    "delinquent_months_formula_equality", "max_delinquency_formula_equality",
    "severe_months_formula_equality", "both_count_ranges_0_to_6",
    "severe_count_le_delinquent_count", "zero_delinquency_implies_zero_max",
    "severe_count_positive_implies_max_ge_2", "max_delinquency_upper_bound",
])
display(delinquency_checks)


,result
delinquent_months_formula_equality,PASS
max_delinquency_formula_equality,PASS
severe_months_formula_equality,PASS
both_count_ranges_0_to_6,PASS
severe_count_le_delinquent_count,PASS
zero_delinquency_implies_zero_max,PASS
severe_count_positive_implies_max_ge_2,PASS
max_delinquency_upper_bound,PASS — maximum observed raw positive PAY code = 8


## 9. Utilization formula sanity checks

The frozen formulas are independently recomputed from the untouched raw BILL values. Values above 1 are expected to remain where bill exceeds limit.


In [10]:
recalc_positive_bill = train_original[BILL_COLS].clip(lower=0)
recalc_current = recalc_positive_bill["BILL_AMT1"] / train_original["LIMIT_BAL"]
recalc_avg = recalc_positive_bill.mean(axis=1) / train_original["LIMIT_BAL"]
recalc_max_util = recalc_positive_bill.max(axis=1) / train_original["LIMIT_BAL"]
util_cols = ["current_utilization", "avg_utilization", "max_utilization"]

pd.testing.assert_series_equal(train_work["current_utilization"], recalc_current, check_names=False, check_exact=True)
pd.testing.assert_series_equal(train_work["avg_utilization"], recalc_avg, check_names=False, check_exact=True)
pd.testing.assert_series_equal(train_work["max_utilization"], recalc_max_util, check_names=False, check_exact=True)
assert (train_original["LIMIT_BAL"] > 0).all()
assert train_work[util_cols].notna().all().all()
assert np.isfinite(train_work[util_cols].to_numpy()).all()
assert (train_work[util_cols] >= 0).all().all()
assert (train_work["current_utilization"] <= train_work["max_utilization"] + 1e-15).all()
assert (train_work["avg_utilization"] <= train_work["max_utilization"] + 1e-15).all()
assert (train_original[BILL_COLS] < 0).any().any()
assert train_work[BILL_COLS].equals(train_original[BILL_COLS])

util_over_one_by_feature = pd.DataFrame({
    "count_gt_1": (train_work[util_cols] > 1).sum(),
    "proportion_gt_1": (train_work[util_cols] > 1).mean(),
})
rows_any_util_over_one = (train_work[util_cols] > 1).any(axis=1)
rows_any_util_over_one_count = int(rows_any_util_over_one.sum())
rows_any_util_over_one_proportion = float(rows_any_util_over_one.mean())
assert rows_any_util_over_one_count > 0

utilization_checks = pd.DataFrame({
    "result": [
        "PASS", "PASS", "PASS", "PASS", "PASS", "PASS", "PASS",
        f"PASS — {rows_any_util_over_one_count:,} rows retained with at least one utilization > 1",
        "PASS — raw BILL values, including negatives, unchanged",
    ]
}, index=[
    "current_formula_equality", "average_formula_equality", "maximum_formula_equality",
    "LIMIT_BAL_strictly_positive", "no_NaN_or_infinity", "all_utilization_nonnegative",
    "current_and_average_le_maximum", "values_above_one_retained", "raw_BILL_integrity",
])
display(utilization_checks)
display(util_over_one_by_feature)
print("Rows with any utilization > 1:", rows_any_util_over_one_count)
print("Proportion with any utilization > 1:", f"{rows_any_util_over_one_proportion:.8%}")
del recalc_positive_bill


,result
current_formula_equality,PASS
average_formula_equality,PASS
maximum_formula_equality,PASS
LIMIT_BAL_strictly_positive,PASS
no_NaN_or_infinity,PASS
all_utilization_nonnegative,PASS
current_and_average_le_maximum,PASS
values_above_one_retained,"PASS — 3,129 rows retained with at least one u..."
raw_BILL_integrity,"PASS — raw BILL values, including negatives, u..."


,count_gt_1,proportion_gt_1
current_utilization,1685,0.070208
avg_utilization,508,0.021167
max_utilization,3129,0.130375


Rows with any utilization > 1: 3129
Proportion with any utilization > 1: 13.03750000%


## 10. Deterministic PAY calculation samples

Rows are selected by fixed conditions and the smallest eligible ID, not by unseeded sampling. They cover no positive month, a single code-1 month, repeated code-2 months, severity 3/4, and severity at least 5.


In [11]:
pay_values = train_work[PAY_COLS]
pay_conditions = [
    ("all six PAY <= 0", (pay_values > 0).sum(axis=1) == 0),
    ("one positive month, max=1", ((pay_values > 0).sum(axis=1) == 1) & (pay_values.max(axis=1) == 1)),
    ("multiple positive months, max=2", ((pay_values > 0).sum(axis=1) >= 2) & (pay_values.max(axis=1) == 2)),
    ("maximum severity 3 or 4", pay_values.max(axis=1).isin([3, 4])),
    ("maximum severity at least 5", pay_values.max(axis=1) >= 5),
]
selected_pay_indices = []
pay_case_labels = {}
for label, mask in pay_conditions:
    candidates = train_work.loc[mask & ~train_work.index.isin(selected_pay_indices)].sort_values(ID_COL)
    assert not candidates.empty
    idx = candidates.index[0]
    selected_pay_indices.append(idx)
    pay_case_labels[idx] = label

pay_sample = train_work.loc[selected_pay_indices, [ID_COL, *PAY_COLS, *NEW_FEATURES[:3]]].copy()
pay_sample.insert(0, "sample_case", [pay_case_labels[idx] for idx in selected_pay_indices])
pay_sample = pay_sample.reset_index(drop=True)
assert len(pay_sample) == len(pay_conditions)
display(pay_sample)


,sample_case,ID,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,delinquent_months,max_delinquency,severe_delinquent_months
0,all six PAY <= 0,3,0,0,0,0,0,0,0,0,0
1,"one positive month, max=1",19,1,-2,-2,-2,-2,-2,1,1,0
2,"multiple positive months, max=2",1,2,2,-1,-1,-2,-2,2,2,2
3,maximum severity 3 or 4,59,-1,3,2,2,2,2,5,3,5
4,maximum severity at least 5,190,1,5,4,4,3,2,6,5,5


## 11. Deterministic BILL/utilization samples

Rows are selected by fixed conditions and smallest eligible ID. Raw BILL values are shown beside utilization so the floor-to-zero calculation is transparent without changing the source columns.


In [12]:
bill_values = train_work[BILL_COLS]
util_conditions = [
    ("at least one negative BILL", (bill_values < 0).any(axis=1)),
    ("at least one zero BILL", (bill_values == 0).any(axis=1)),
    ("ordinary positive bills, none above limit", (bill_values > 0).all(axis=1) & bill_values.le(train_work["LIMIT_BAL"], axis=0).all(axis=1)),
    ("at least one BILL above LIMIT_BAL", bill_values.gt(train_work["LIMIT_BAL"], axis=0).any(axis=1)),
]
selected_util_indices = []
util_case_labels = {}
for label, mask in util_conditions:
    candidates = train_work.loc[mask & ~train_work.index.isin(selected_util_indices)].sort_values(ID_COL)
    assert not candidates.empty
    idx = candidates.index[0]
    selected_util_indices.append(idx)
    util_case_labels[idx] = label

util_sample = train_work.loc[selected_util_indices, [ID_COL, "LIMIT_BAL", *BILL_COLS, *NEW_FEATURES[3:]]].copy()
util_sample.insert(0, "sample_case", [util_case_labels[idx] for idx in selected_util_indices])
util_sample = util_sample.reset_index(drop=True)
assert len(util_sample) == len(util_conditions)
display(util_sample)


,sample_case,ID,LIMIT_BAL,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,current_utilization,avg_utilization,max_utilization
0,at least one negative BILL,8,100000.0,11876.0,380.0,601.0,221.0,-159.0,567.0,0.118760,0.022742,0.118760
1,at least one zero BILL,1,20000.0,3913.0,3102.0,689.0,0.0,0.0,0.0,0.195650,0.064200,0.195650
2,"ordinary positive bills, none above limit",3,90000.0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,0.324878,0.188246,0.324878
3,at least one BILL above LIMIT_BAL,6,50000.0,64400.0,57069.0,57608.0,19394.0,19619.0,20024.0,1.288000,0.793713,1.288000


## 12. Train-only descriptive sanity outputs for the six new features

These outputs contain no target comparison, default-rate analysis, correlation, IV, AUC, or ranking. They describe only the fixed features for sanity and human understanding.


In [13]:
new_feature_summary = train_work[NEW_FEATURES].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T
new_feature_zero_summary = pd.DataFrame({
    "zero_count": (train_work[NEW_FEATURES] == 0).sum(),
    "zero_proportion": (train_work[NEW_FEATURES] == 0).mean(),
})

delinquency_value_counts = pd.concat(
    {
        col: train_work[col].value_counts().sort_index()
        for col in NEW_FEATURES[:3]
    },
    axis=1,
).fillna(0).astype(int)

print("Continuous/discrete numeric summary (Train only; no target comparison):")
display(new_feature_summary)
print("Zero frequencies (Train only):")
display(new_feature_zero_summary)
print("Discrete delinquency value counts:")
display(delinquency_value_counts)
print("Utilization >1 frequencies:")
display(util_over_one_by_feature)


Continuous/discrete numeric summary (Train only; no target comparison):


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
delinquent_months,24000.0,0.835708,1.550828,0.0,0.0,0.000000,0.000000,0.000000,1.000000,5.000000,6.000000,6.000000
max_delinquency,24000.0,0.682875,1.066937,0.0,0.0,0.000000,0.000000,0.000000,2.000000,2.000000,4.000000,8.000000
severe_delinquent_months,24000.0,0.709458,1.459640,0.0,0.0,0.000000,0.000000,0.000000,1.000000,5.000000,6.000000,6.000000
current_utilization,24000.0,0.423506,0.410234,0.0,0.0,0.000000,0.021788,0.314936,0.828832,1.012805,1.233520,6.455300
avg_utilization,24000.0,0.372518,0.350969,0.0,0.0,0.001157,0.030519,0.284486,0.686247,0.959133,1.048268,5.364308
max_utilization,24000.0,0.494320,0.432641,0.0,0.0,0.003225,0.071571,0.429782,0.921900,1.033013,1.565131,10.688575


Zero frequencies (Train only):


,zero_count,zero_proportion
delinquent_months,15896,0.662333
max_delinquency,15896,0.662333
severe_delinquent_months,17276,0.719833
current_utilization,2080,0.086667
avg_utilization,771,0.032125
max_utilization,771,0.032125


Discrete delinquency value counts:


,delinquent_months,max_delinquency,severe_delinquent_months
0,15896,15896,17276
1,3572,1380,2799
2,1532,5785,1279
3,936,629,861
4,767,173,570
5,237,49,483
6,1060,20,732
7,0,49,0
8,0,19,0


Utilization >1 frequencies:


,count_gt_1,proportion_gt_1
current_utilization,1685,0.070208
avg_utilization,508,0.021167
max_utilization,3129,0.130375


## 13. Scope confirmation and Phase A Human Checkpoint

The frozen Train rules have been implemented and checked. Test application remains pending Architect/user confirmation. This Notebook does not complete Stage 3.3 and must stop before Phase B.


In [14]:
assert TEST_ISOLATION_STATUS.startswith("SEALED")
assert set(train_work.columns) - set(train_original.columns) == set(GROUP_FIELDS + NEW_FEATURES)
assert X_train.columns.tolist() == PREDICTOR_COLUMNS
assert len(GROUP_FIELDS) == 2
assert len(NEW_FEATURES) == 6
assert "AgeBin" not in train_work.columns
assert "SE_MA" not in train_work.columns
assert not any(col.startswith("Avg_exp") or col.startswith("Closeness") for col in train_work.columns)

scope_status = pd.DataFrame({
    "result": [
        "PASS — Test not transformed and not retained in live state",
        "PASS — exactly two frozen group fields",
        "PASS — exactly six approved new features",
        "PASS — no unauthorized feature",
        "PASS — no target-driven feature reassessment",
        "PASS — no WOE, binning, modeling, CV, sampling, or imbalance handling",
        "PASS — Phase B pending; Human Checkpoint reached",
    ]
}, index=[
    "Test_isolation", "category_scope", "new_feature_scope", "unauthorized_features",
    "selection_scope", "modeling_and_imbalance_scope", "stop_condition",
])
display(scope_status)
print("Stage 3.3 — Phase A completed; Test application pending Architect/user confirmation.")


,result
Test_isolation,PASS — Test not transformed and not retained i...
category_scope,PASS — exactly two frozen group fields
new_feature_scope,PASS — exactly six approved new features
unauthorized_features,PASS — no unauthorized feature
selection_scope,PASS — no target-driven feature reassessment
modeling_and_imbalance_scope,"PASS — no WOE, binning, modeling, CV, sampling..."
stop_condition,PASS — Phase B pending; Human Checkpoint reached


Stage 3.3 — Phase A completed; Test application pending Architect/user confirmation.


# Stage 3.3 — Phase B
# Apply Frozen Feature Engineering to Holdout Test + Final Sanity Check

## Authorization and scope

The Architect/user Human Checkpoint authorizes applying the already-approved Phase A deterministic rules to the Holdout Test. No feature is redesigned or added. Test outcome information remains sealed: it is retained only as an untouched, separately aligned label and is never supplied to transformation, summaries, samples, grouping, selection, or evaluation.


In [15]:
# Independently reload raw data and replay the locked split for Phase B.
phase_b_raw = pd.read_csv(data_path)
assert phase_b_raw.shape == (30000, 25)
assert phase_b_raw[ID_COL].is_unique
assert phase_b_raw.isna().sum().sum() == 0

phase_b_train_raw, phase_b_test_raw = train_test_split(
    phase_b_raw,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=phase_b_raw[TARGET],
)

phase_b_train_ids = set(phase_b_train_raw[ID_COL].astype(int))
phase_b_test_ids = set(phase_b_test_raw[ID_COL].astype(int))
phase_b_full_ids = set(phase_b_raw[ID_COL].astype(int))
phase_b_train_hash = stable_id_set_hash(phase_b_train_ids)
phase_b_test_hash = stable_id_set_hash(phase_b_test_ids)

assert len(phase_b_train_raw) == 24000
assert len(phase_b_test_raw) == 6000
assert phase_b_train_ids.isdisjoint(phase_b_test_ids)
assert phase_b_train_ids | phase_b_test_ids == phase_b_full_ids
assert len(phase_b_train_ids) + len(phase_b_test_ids) == 30000
assert phase_b_train_hash == EXPECTED_TRAIN_HASH
assert phase_b_test_hash == EXPECTED_TEST_HASH

# Labels are isolated before transformation. The transformation input contains no target.
phase_b_train_trace = phase_b_train_raw[[ID_COL, TARGET]].copy(deep=True)
phase_b_test_trace = phase_b_test_raw[[ID_COL, TARGET]].copy(deep=True)
phase_b_test_trace_before = phase_b_test_trace.copy(deep=True)
phase_b_train_input = phase_b_train_raw.drop(columns=[TARGET]).copy(deep=True)
phase_b_test_input = phase_b_test_raw.drop(columns=[TARGET]).copy(deep=True)
assert TARGET not in phase_b_train_input.columns
assert TARGET not in phase_b_test_input.columns
assert phase_b_train_input.index.equals(phase_b_train_trace.index)
assert phase_b_test_input.index.equals(phase_b_test_trace.index)

phase_b_split_evidence = pd.DataFrame({
    "value": [
        "train_test_split", "80% / 20%", RANDOM_STATE, TARGET,
        len(phase_b_raw), len(phase_b_train_input), len(phase_b_test_input),
        len(phase_b_train_ids & phase_b_test_ids),
        phase_b_train_ids | phase_b_test_ids == phase_b_full_ids,
        phase_b_train_hash, phase_b_test_hash,
    ]
}, index=[
    "method", "ratio", "random_state", "stratification_field_name",
    "raw_rows", "train_rows", "test_rows", "ID_overlap", "full_ID_coverage",
    "train_ID_SHA256", "test_ID_SHA256",
])
display(phase_b_split_evidence)
print("Phase B split replay passed before Test transformation.")
del phase_b_raw, phase_b_train_raw, phase_b_test_raw, phase_b_full_ids


,value
method,train_test_split
ratio,80% / 20%
random_state,42
stratification_field_name,default.payment.next.month
raw_rows,30000
train_rows,24000
test_rows,6000
ID_overlap,0
full_ID_coverage,True
train_ID_SHA256,bcb37af63ae7005df78e6140837594b5d650d27148b46b...


Phase B split replay passed before Test transformation.


## 14. Test-target protection contract and one frozen transformation

- The Test label is stored only in `phase_b_test_trace`; the transformation function cannot receive it.
- Test labels are used only for exact value and ID-alignment preservation assertions.
- No Test target count, rate, grouping, correlation, plot, metric, or other target-based analysis is computed or displayed.
- One deterministic implementation applies the Phase A mappings and six formulas identically to Train and Test.


In [16]:
RAW_PREDICTOR_PRESERVED_COLUMNS = [
    ID_COL, "AGE", "SEX", "LIMIT_BAL", "EDUCATION", "MARRIAGE",
    *PAY_COLS, *BILL_COLS, *PAY_AMT_COLS,
]

def apply_frozen_feature_engineering(raw_predictor_input):
    """Apply only the Phase A-approved deterministic rules; target is not accepted."""
    assert TARGET not in raw_predictor_input.columns
    assert ID_COL in raw_predictor_input.columns
    assert set(raw_predictor_input["EDUCATION"].unique()).issubset(EDUCATION_MAP)
    assert set(raw_predictor_input["MARRIAGE"].unique()).issubset(MARRIAGE_MAP)
    assert (raw_predictor_input["LIMIT_BAL"] > 0).all()

    work = raw_predictor_input.copy(deep=True)
    work["EDUCATION_GROUP"] = work["EDUCATION"].map(EDUCATION_MAP)
    work["MARRIAGE_GROUP"] = work["MARRIAGE"].map(MARRIAGE_MAP)
    assert work[GROUP_FIELDS].notna().all().all()

    raw_pay = work[PAY_COLS]
    work["delinquent_months"] = (raw_pay > 0).sum(axis=1).astype("int64")
    work["max_delinquency"] = raw_pay.clip(lower=0).max(axis=1).astype("int64")
    work["severe_delinquent_months"] = (raw_pay >= 2).sum(axis=1).astype("int64")

    calculation_only_positive_bill = work[BILL_COLS].clip(lower=0)
    work["current_utilization"] = calculation_only_positive_bill["BILL_AMT1"] / work["LIMIT_BAL"]
    work["avg_utilization"] = calculation_only_positive_bill.mean(axis=1) / work["LIMIT_BAL"]
    work["max_utilization"] = calculation_only_positive_bill.max(axis=1) / work["LIMIT_BAL"]
    del calculation_only_positive_bill

    predictors = work[PREDICTOR_COLUMNS].copy()
    assert predictors.columns.tolist() == PREDICTOR_COLUMNS
    assert len(predictors.columns) == 29 and predictors.columns.is_unique
    return work, predictors

phase_b_train_work, phase_b_X_train = apply_frozen_feature_engineering(phase_b_train_input)
phase_b_test_work, X_test = apply_frozen_feature_engineering(phase_b_test_input)

# Target remains separate and is only re-aligned by the untouched original index.
phase_b_test_trace_after = phase_b_test_trace.loc[phase_b_test_work.index, [ID_COL, TARGET]].copy()
pd.testing.assert_frame_equal(
    phase_b_test_trace_after, phase_b_test_trace_before,
    check_dtype=True, check_exact=True,
)

display(pd.DataFrame({"result": [
    "PASS — same frozen function applied to Train and Test",
    "PASS — target excluded from transformation inputs and predictors",
    "PASS — Test label values and ID alignment unchanged",
]}, index=["single_frozen_definition", "target_exclusion", "target_integrity"]))


,result
single_frozen_definition,PASS — same frozen function applied to Train a...
target_exclusion,PASS — target excluded from transformation inp...
target_integrity,PASS — Test label values and ID alignment unch...


## 15. Train regression validation

Phase B must not change any accepted Phase A Train result. The common frozen implementation is therefore compared exactly with the saved Phase A Train structures, summaries, counts, and deterministic sample rows. This is regression validation only, not feature reassessment.


In [17]:
assert len(phase_b_train_work) == 24000
assert stable_id_set_hash(phase_b_train_work[ID_COL].astype(int)) == EXPECTED_TRAIN_HASH
pd.testing.assert_frame_equal(phase_b_X_train, X_train, check_dtype=True, check_exact=True)
pd.testing.assert_frame_equal(
    phase_b_train_work[RAW_PREDICTOR_PRESERVED_COLUMNS],
    train_original[RAW_PREDICTOR_PRESERVED_COLUMNS],
    check_dtype=True, check_exact=True,
)

phase_b_train_education_counts = phase_b_train_work["EDUCATION_GROUP"].value_counts().sort_index()
phase_b_train_marriage_counts = phase_b_train_work["MARRIAGE_GROUP"].value_counts().sort_index()
assert phase_b_train_education_counts.equals(education_group_counts)
assert phase_b_train_marriage_counts.equals(marriage_group_counts)
assert int(phase_b_train_education_counts["Other / Unknown"]) == 386
assert int(phase_b_train_marriage_counts["Other / Unknown"]) == 302
assert int((phase_b_train_work["delinquent_months"] == 0).sum()) == 15896
assert int((phase_b_train_work["max_delinquency"] == 0).sum()) == 15896
assert int((phase_b_train_work["severe_delinquent_months"] == 0).sum()) == 17276
assert int((phase_b_train_work[util_cols] > 1).any(axis=1).sum()) == 3129

phase_b_train_summary = phase_b_train_work[NEW_FEATURES].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T
pd.testing.assert_frame_equal(phase_b_train_summary, new_feature_summary, check_exact=True)

phase_b_pay_sample = phase_b_train_work.loc[
    selected_pay_indices, [ID_COL, *PAY_COLS, *NEW_FEATURES[:3]]
].copy()
phase_b_pay_sample.insert(0, "sample_case", [pay_case_labels[idx] for idx in selected_pay_indices])
phase_b_pay_sample = phase_b_pay_sample.reset_index(drop=True)
pd.testing.assert_frame_equal(phase_b_pay_sample, pay_sample, check_dtype=True, check_exact=True)

phase_b_util_sample = phase_b_train_work.loc[
    selected_util_indices, [ID_COL, "LIMIT_BAL", *BILL_COLS, *NEW_FEATURES[3:]]
].copy()
phase_b_util_sample.insert(0, "sample_case", [util_case_labels[idx] for idx in selected_util_indices])
phase_b_util_sample = phase_b_util_sample.reset_index(drop=True)
pd.testing.assert_frame_equal(phase_b_util_sample, util_sample, check_dtype=True, check_exact=True)

assert phase_b_X_train.isna().sum().sum() == 0
assert np.isfinite(phase_b_X_train.select_dtypes(include=np.number).to_numpy()).all()

train_regression_results = pd.DataFrame({"actual": [
    len(phase_b_train_work), phase_b_train_hash, phase_b_X_train.shape[1],
    int(phase_b_train_education_counts["Other / Unknown"]),
    int(phase_b_train_marriage_counts["Other / Unknown"]),
    int((phase_b_train_work["delinquent_months"] == 0).sum()),
    int((phase_b_train_work["max_delinquency"] == 0).sum()),
    int((phase_b_train_work["severe_delinquent_months"] == 0).sum()),
    int((phase_b_train_work[util_cols] > 1).any(axis=1).sum()),
    "PASS — all six summaries and deterministic samples unchanged",
]}, index=[
    "Train_rows", "Train_ID_SHA256", "predictor_count",
    "EDUCATION_Other_Unknown", "MARRIAGE_Other_Unknown",
    "delinquent_months_zero", "max_delinquency_zero",
    "severe_delinquent_months_zero", "rows_any_utilization_gt_1",
    "Phase_A_summary_and_sample_regression",
])
display(train_regression_results)


,actual
Train_rows,24000
Train_ID_SHA256,bcb37af63ae7005df78e6140837594b5d650d27148b46b...
predictor_count,29
EDUCATION_Other_Unknown,386
MARRIAGE_Other_Unknown,302
delinquent_months_zero,15896
max_delinquency_zero,15896
severe_delinquent_months_zero,17276
rows_any_utilization_gt_1,3129
Phase_A_summary_and_sample_regression,PASS — all six summaries and deterministic sam...


## 16. Holdout Test structural integrity and predictor-ready schema

All 6,000 Holdout Test rows retain their original ID order, raw predictor values, and separately sealed labels. The predictor view must match the frozen 29-column Train schema exactly, with compatible dtypes and no calculation-only or unauthorized field.


In [18]:
pd.testing.assert_frame_equal(
    phase_b_test_work[RAW_PREDICTOR_PRESERVED_COLUMNS],
    phase_b_test_input[RAW_PREDICTOR_PRESERVED_COLUMNS],
    check_dtype=True, check_exact=True,
)
assert len(phase_b_test_work) == len(phase_b_test_input) == 6000
assert phase_b_test_work.index.equals(phase_b_test_input.index)
assert phase_b_test_work[ID_COL].equals(phase_b_test_input[ID_COL])
assert phase_b_test_work[ID_COL].is_unique
assert phase_b_test_work.duplicated(subset=[ID_COL]).sum() == 0
assert set(phase_b_test_work[ID_COL].astype(int)) == phase_b_test_ids
assert stable_id_set_hash(phase_b_test_work[ID_COL].astype(int)) == EXPECTED_TEST_HASH
pd.testing.assert_frame_equal(phase_b_test_trace_after, phase_b_test_trace_before, check_exact=True)

assert X_test.shape == (6000, 29)
assert phase_b_X_train.shape == (24000, 29)
assert X_test.columns.tolist() == PREDICTOR_COLUMNS
assert phase_b_X_train.columns.tolist() == PREDICTOR_COLUMNS
assert X_test.columns.tolist() == phase_b_X_train.columns.tolist()
assert X_test.columns.is_unique and phase_b_X_train.columns.is_unique
assert ID_COL not in X_test.columns and TARGET not in X_test.columns
assert "EDUCATION" not in X_test.columns and "MARRIAGE" not in X_test.columns
assert not any(col.startswith("positive_bill") for col in X_test.columns)
assert set(phase_b_test_work.columns) - set(phase_b_test_input.columns) == set(GROUP_FIELDS + NEW_FEATURES)
assert set(phase_b_train_work.columns) - set(phase_b_train_input.columns) == set(GROUP_FIELDS + NEW_FEATURES)

dtype_compatibility = pd.Series({
    col: phase_b_X_train[col].dtype == X_test[col].dtype for col in PREDICTOR_COLUMNS
})
assert dtype_compatibility.all()
assert X_test.isna().sum().sum() == 0
assert np.isfinite(X_test.select_dtypes(include=np.number).to_numpy()).all()

schema_equality_table = pd.DataFrame({
    "position": range(1, 30),
    "predictor": PREDICTOR_COLUMNS,
    "Train_dtype": [str(phase_b_X_train[c].dtype) for c in PREDICTOR_COLUMNS],
    "Test_dtype": [str(X_test[c].dtype) for c in PREDICTOR_COLUMNS],
    "dtype_compatible": [bool(dtype_compatibility[c]) for c in PREDICTOR_COLUMNS],
})
display(schema_equality_table)
display(pd.DataFrame({"result": [
    "PASS — 6,000 Test rows; no loss or duplication",
    "PASS — Test ID order/set unchanged and unique",
    "PASS — target values and ID alignment unchanged",
    "PASS — all required raw fields unchanged",
    "PASS — identical ordered 29-column Train/Test schema",
    "PASS — corresponding Train/Test dtypes compatible",
    "PASS — no NaN or numeric infinity",
]}, index=[
    "row_integrity", "ID_integrity", "target_integrity", "raw_field_integrity",
    "schema_equality", "dtype_compatibility", "complete_finite_predictors",
]))


,position,predictor,Train_dtype,Test_dtype,dtype_compatible
0,1,LIMIT_BAL,float64,float64,True
1,2,SEX,int64,int64,True
2,3,EDUCATION_GROUP,object,object,True
3,4,MARRIAGE_GROUP,object,object,True
4,5,AGE,int64,int64,True
5,6,PAY_0,int64,int64,True
6,7,PAY_2,int64,int64,True
7,8,PAY_3,int64,int64,True
8,9,PAY_4,int64,int64,True
9,10,PAY_5,int64,int64,True


,result
row_integrity,"PASS — 6,000 Test rows; no loss or duplication"
ID_integrity,PASS — Test ID order/set unchanged and unique
target_integrity,PASS — target values and ID alignment unchanged
raw_field_integrity,PASS — all required raw fields unchanged
schema_equality,PASS — identical ordered 29-column Train/Test ...
dtype_compatibility,PASS — corresponding Train/Test dtypes compatible
complete_finite_predictors,PASS — no NaN or numeric infinity


## 17. Test category mapping validation

These non-target reconciliation tables verify only that every raw category is inside the frozen domain and maps exactly once to an allowed group. They are not compared with the Test target or used to revise any rule.


In [19]:
assert set(phase_b_test_input["EDUCATION"].unique()).issubset(EDUCATION_MAP)
assert set(phase_b_test_input["MARRIAGE"].unique()).issubset(MARRIAGE_MAP)
assert set(phase_b_test_work["EDUCATION_GROUP"].unique()).issubset(EXPECTED_EDUCATION_GROUPS)
assert set(phase_b_test_work["MARRIAGE_GROUP"].unique()).issubset(EXPECTED_MARRIAGE_GROUPS)
assert phase_b_test_work[GROUP_FIELDS].notna().all().all()

test_education_reconciliation = (
    phase_b_test_work.groupby(["EDUCATION", "EDUCATION_GROUP"], observed=True)
    .size().rename("Test_count").reset_index().sort_values("EDUCATION")
)
test_marriage_reconciliation = (
    phase_b_test_work.groupby(["MARRIAGE", "MARRIAGE_GROUP"], observed=True)
    .size().rename("Test_count").reset_index().sort_values("MARRIAGE")
)
test_education_group_counts = phase_b_test_work["EDUCATION_GROUP"].value_counts().sort_index()
test_marriage_group_counts = phase_b_test_work["MARRIAGE_GROUP"].value_counts().sort_index()

assert test_education_reconciliation["Test_count"].sum() == 6000
assert test_marriage_reconciliation["Test_count"].sum() == 6000
assert test_education_group_counts.sum() == 6000
assert test_marriage_group_counts.sum() == 6000
assert test_education_group_counts.equals(
    phase_b_test_input["EDUCATION"].map(EDUCATION_MAP).value_counts().sort_index()
)
assert test_marriage_group_counts.equals(
    phase_b_test_input["MARRIAGE"].map(MARRIAGE_MAP).value_counts().sort_index()
)

print("Test EDUCATION raw-code reconciliation (no target):")
display(test_education_reconciliation)
print("Test EDUCATION grouped counts (no target):")
display(test_education_group_counts.rename("Test_count").to_frame())
print("Test MARRIAGE raw-code reconciliation (no target):")
display(test_marriage_reconciliation)
print("Test MARRIAGE grouped counts (no target):")
display(test_marriage_group_counts.rename("Test_count").to_frame())


Test EDUCATION raw-code reconciliation (no target):


,EDUCATION,EDUCATION_GROUP,Test_count
0,0,Other / Unknown,2
1,1,Graduate School,2130
2,2,University,2774
3,3,High School,1014
4,4,Other / Unknown,26
5,5,Other / Unknown,45
6,6,Other / Unknown,9


Test EDUCATION grouped counts (no target):


,Test_count
EDUCATION_GROUP,
Graduate School,2130
High School,1014
Other / Unknown,82
University,2774


Test MARRIAGE raw-code reconciliation (no target):


,MARRIAGE,MARRIAGE_GROUP,Test_count
0,0,Other / Unknown,7
1,1,Married,2767
2,2,Single,3158
3,3,Other / Unknown,68


Test MARRIAGE grouped counts (no target):


,Test_count
MARRIAGE_GROUP,
Married,2767
Other / Unknown,75
Single,3158


## 18. Test delinquency arithmetic validation

The three approved delinquency features are independently recalculated from untouched Test PAY fields. Deterministic samples are selected by fixed non-target conditions and show no outcome label.


In [20]:
test_recalc_delinquent = (phase_b_test_input[PAY_COLS] > 0).sum(axis=1).astype("int64")
test_recalc_max = phase_b_test_input[PAY_COLS].clip(lower=0).max(axis=1).astype("int64")
test_recalc_severe = (phase_b_test_input[PAY_COLS] >= 2).sum(axis=1).astype("int64")

pd.testing.assert_series_equal(
    phase_b_test_work["delinquent_months"], test_recalc_delinquent, check_names=False, check_exact=True
)
pd.testing.assert_series_equal(
    phase_b_test_work["max_delinquency"], test_recalc_max, check_names=False, check_exact=True
)
pd.testing.assert_series_equal(
    phase_b_test_work["severe_delinquent_months"], test_recalc_severe, check_names=False, check_exact=True
)
assert phase_b_test_work["delinquent_months"].between(0, 6).all()
assert phase_b_test_work["severe_delinquent_months"].between(0, 6).all()
assert (phase_b_test_work["severe_delinquent_months"] <= phase_b_test_work["delinquent_months"]).all()
assert (phase_b_test_work["max_delinquency"] >= 0).all()
assert (phase_b_test_work.loc[
    phase_b_test_work["delinquent_months"] == 0, "max_delinquency"
] == 0).all()
assert (phase_b_test_work.loc[
    phase_b_test_work["severe_delinquent_months"] > 0, "max_delinquency"
] >= 2).all()
assert phase_b_test_work[PAY_COLS].equals(phase_b_test_input[PAY_COLS])

test_pay_values = phase_b_test_work[PAY_COLS]
test_pay_conditions = [
    ("all six PAY <= 0", (test_pay_values > 0).sum(axis=1) == 0),
    ("one positive month, max=1", ((test_pay_values > 0).sum(axis=1) == 1) & (test_pay_values.max(axis=1) == 1)),
    ("multiple positive months, max=2", ((test_pay_values > 0).sum(axis=1) >= 2) & (test_pay_values.max(axis=1) == 2)),
    ("maximum severity 3 or 4", test_pay_values.max(axis=1).isin([3, 4])),
    ("maximum severity at least 5", test_pay_values.max(axis=1) >= 5),
]
test_selected_pay_indices = []
test_pay_case_labels = {}
for label, mask in test_pay_conditions:
    candidates = phase_b_test_work.loc[
        mask & ~phase_b_test_work.index.isin(test_selected_pay_indices)
    ].sort_values(ID_COL)
    assert not candidates.empty
    idx = candidates.index[0]
    test_selected_pay_indices.append(idx)
    test_pay_case_labels[idx] = label

test_pay_sample = phase_b_test_work.loc[
    test_selected_pay_indices, [ID_COL, *PAY_COLS, *NEW_FEATURES[:3]]
].copy()
test_pay_sample.insert(0, "sample_case", [test_pay_case_labels[i] for i in test_selected_pay_indices])
test_pay_sample = test_pay_sample.reset_index(drop=True)
assert TARGET not in test_pay_sample.columns and len(test_pay_sample) == 5

display(pd.DataFrame({"result": [
    "PASS — all three formulas independently reproduced",
    "PASS — ranges and cross-feature logic valid",
    "PASS — six raw PAY fields unchanged",
    "PASS — five deterministic non-target arithmetic samples validated",
]}, index=["formula_equality", "logical_assertions", "raw_PAY_integrity", "sample_validation"]))
display(test_pay_sample)


,result
formula_equality,PASS — all three formulas independently reprod...
logical_assertions,PASS — ranges and cross-feature logic valid
raw_PAY_integrity,PASS — six raw PAY fields unchanged
sample_validation,PASS — five deterministic non-target arithmeti...


,sample_case,ID,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,delinquent_months,max_delinquency,severe_delinquent_months
0,all six PAY <= 0,15,0,0,0,0,0,0,0,0,0
1,"one positive month, max=1",27,1,-2,-1,-1,-1,-1,1,1,0
2,"multiple positive months, max=2",2,-1,2,0,0,0,2,2,2,2
3,maximum severity 3 or 4,534,2,0,0,3,3,2,4,3,4
4,maximum severity at least 5,159,2,3,4,4,5,6,6,6,6


## 19. Test utilization arithmetic validation

The three approved utilization features are independently recalculated. Negative BILL values remain raw and are floored only in a temporary calculation matrix. Values above 1 are retained. Deterministic samples contain no target.


In [21]:
assert (phase_b_test_input["LIMIT_BAL"] > 0).all()
test_recalc_positive_bill = phase_b_test_input[BILL_COLS].clip(lower=0)
test_recalc_current = test_recalc_positive_bill["BILL_AMT1"] / phase_b_test_input["LIMIT_BAL"]
test_recalc_avg = test_recalc_positive_bill.mean(axis=1) / phase_b_test_input["LIMIT_BAL"]
test_recalc_max_util = test_recalc_positive_bill.max(axis=1) / phase_b_test_input["LIMIT_BAL"]

pd.testing.assert_series_equal(
    phase_b_test_work["current_utilization"], test_recalc_current, check_names=False, check_exact=True
)
pd.testing.assert_series_equal(
    phase_b_test_work["avg_utilization"], test_recalc_avg, check_names=False, check_exact=True
)
pd.testing.assert_series_equal(
    phase_b_test_work["max_utilization"], test_recalc_max_util, check_names=False, check_exact=True
)
assert phase_b_test_work[util_cols].notna().all().all()
assert np.isfinite(phase_b_test_work[util_cols].to_numpy()).all()
assert (phase_b_test_work[util_cols] >= 0).all().all()
assert (phase_b_test_work["current_utilization"] <= phase_b_test_work["max_utilization"] + 1e-15).all()
assert (phase_b_test_work["avg_utilization"] <= phase_b_test_work["max_utilization"] + 1e-15).all()
assert phase_b_test_work[BILL_COLS].equals(phase_b_test_input[BILL_COLS])
assert (phase_b_test_input[BILL_COLS] < 0).any().any()

test_util_over_one_by_feature = pd.DataFrame({
    "count_gt_1": (phase_b_test_work[util_cols] > 1).sum(),
})
test_rows_any_util_over_one = int((phase_b_test_work[util_cols] > 1).any(axis=1).sum())
assert test_rows_any_util_over_one > 0

test_bill_values = phase_b_test_work[BILL_COLS]
test_util_conditions = [
    ("at least one negative BILL", (test_bill_values < 0).any(axis=1)),
    ("at least one zero BILL", (test_bill_values == 0).any(axis=1)),
    ("ordinary positive bills, none above limit", (test_bill_values > 0).all(axis=1) & test_bill_values.le(phase_b_test_work["LIMIT_BAL"], axis=0).all(axis=1)),
    ("at least one BILL above LIMIT_BAL", test_bill_values.gt(phase_b_test_work["LIMIT_BAL"], axis=0).any(axis=1)),
]
test_selected_util_indices = []
test_util_case_labels = {}
for label, mask in test_util_conditions:
    candidates = phase_b_test_work.loc[
        mask & ~phase_b_test_work.index.isin(test_selected_util_indices)
    ].sort_values(ID_COL)
    assert not candidates.empty
    idx = candidates.index[0]
    test_selected_util_indices.append(idx)
    test_util_case_labels[idx] = label

test_util_sample = phase_b_test_work.loc[
    test_selected_util_indices, [ID_COL, "LIMIT_BAL", *BILL_COLS, *NEW_FEATURES[3:]]
].copy()
test_util_sample.insert(0, "sample_case", [test_util_case_labels[i] for i in test_selected_util_indices])
test_util_sample = test_util_sample.reset_index(drop=True)
assert TARGET not in test_util_sample.columns and len(test_util_sample) == 4

display(pd.DataFrame({"result": [
    "PASS — all three formulas independently reproduced",
    "PASS — finite, nonnegative, current/average <= maximum",
    "PASS — raw negatives retained; flooring calculation-local",
    f"PASS — {test_rows_any_util_over_one:,} Test rows retain utilization > 1",
    "PASS — four deterministic non-target arithmetic samples validated",
]}, index=[
    "formula_equality", "numeric_assertions", "raw_BILL_integrity",
    "values_above_one_retained", "sample_validation",
]))
display(test_util_over_one_by_feature)
display(test_util_sample)
del test_recalc_positive_bill


,result
formula_equality,PASS — all three formulas independently reprod...
numeric_assertions,"PASS — finite, nonnegative, current/average <=..."
raw_BILL_integrity,PASS — raw negatives retained; flooring calcul...
values_above_one_retained,PASS — 802 Test rows retain utilization > 1
sample_validation,PASS — four deterministic non-target arithmeti...


,count_gt_1
current_utilization,430
avg_utilization,130
max_utilization,802


,sample_case,ID,LIMIT_BAL,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,current_utilization,avg_utilization,max_utilization
0,at least one negative BILL,27,60000.0,-109.0,-425.0,259.0,-57.0,127.0,-189.0,0.000000,0.001072,0.004317
1,at least one zero BILL,22,120000.0,316.0,316.0,316.0,0.0,632.0,316.0,0.002633,0.002633,0.005267
2,"ordinary positive bills, none above limit",2,120000.0,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.022350,0.023718,0.028792
3,at least one BILL above LIMIT_BAL,105,60000.0,56685.0,55208.0,59175.0,60218.0,55447.0,55305.0,0.944750,0.950106,1.003633


## 20. Limited non-target Test sanity summary

The following evidence is intentionally limited to transformation validation: row count, completeness, allowed mappings, derived-feature ranges, and retained utilization-above-one counts. It is not Train/Test drift analysis and contains no target information.


In [22]:
test_limited_sanity = pd.DataFrame({
    "minimum": phase_b_test_work[NEW_FEATURES].min(),
    "maximum": phase_b_test_work[NEW_FEATURES].max(),
    "missing_count": phase_b_test_work[NEW_FEATURES].isna().sum(),
})
test_nonfinite_numeric_count = int((~np.isfinite(X_test.select_dtypes(include=np.number))).sum().sum())
assert test_nonfinite_numeric_count == 0
assert test_limited_sanity["missing_count"].sum() == 0

display(pd.DataFrame({"actual": [
    len(X_test), X_test.shape[1], int(X_test.isna().sum().sum()),
    test_nonfinite_numeric_count, test_rows_any_util_over_one,
]}, index=[
    "Test_rows", "predictor_count", "predictor_missing_cells",
    "numeric_nonfinite_cells", "rows_any_utilization_gt_1",
]))
display(test_limited_sanity)
print("Limited non-target Test sanity evidence only; no drift or predictive analysis performed.")


,actual
Test_rows,6000
predictor_count,29
predictor_missing_cells,0
numeric_nonfinite_cells,0
rows_any_utilization_gt_1,802


,minimum,maximum,missing_count
delinquent_months,0.0,6.000000,0
max_delinquency,0.0,8.000000,0
severe_delinquent_months,0.0,6.000000,0
current_utilization,0.0,4.140600,0
avg_utilization,0.0,3.759125,0
max_utilization,0.0,4.935500,0


Limited non-target Test sanity evidence only; no drift or predictive analysis performed.


## 21. Protected-target confirmation and Stage 3.3 Reviewer handoff

The locked split, Phase A Train results, frozen Test transformation, raw-field integrity, arithmetic formulas, and identical 29-column schemas have now been validated. Test target information remains sealed and untouched. This completes implementation only; Stage 3.3 remains pending independent final Reviewer validation and the separately authorized milestone commit.


In [23]:
pd.testing.assert_frame_equal(phase_b_test_trace, phase_b_test_trace_before, check_exact=True)
pd.testing.assert_frame_equal(phase_b_test_trace_after, phase_b_test_trace_before, check_exact=True)
assert TARGET not in X_test.columns and TARGET not in phase_b_test_input.columns
assert ID_COL not in X_test.columns
assert X_test.columns.tolist() == phase_b_X_train.columns.tolist() == PREDICTOR_COLUMNS
assert len(PREDICTOR_COLUMNS) == 29 and len(NEW_FEATURES) == 6
assert set(NEW_FEATURES) == {
    "delinquent_months", "max_delinquency", "severe_delinquent_months",
    "current_utilization", "avg_utilization", "max_utilization",
}
assert not any(col in X_test.columns for col in ["AgeBin", "SE_MA", "SE_AG"])
assert not any(col.startswith("positive_bill") for col in X_test.columns)

final_stage_3_3_status = pd.DataFrame({"result": [
    "PASS — locked split and both stable ID hashes reproduced",
    "PASS — Phase A Train accepted results unchanged",
    "PASS — frozen rules applied identically to Holdout Test",
    "PASS — Train/Test ordered 29-column schemas identical",
    "PASS — Test ID, rows, raw fields, and sealed labels unchanged",
    "PASS — no new feature or feature redesign",
    "PASS — no Test target analysis or Train/Test drift analysis",
    "PASS — no WOE, IV, binning, imbalance handling, modeling, or evaluation",
    "PENDING — independent final Reviewer validation and milestone commit",
]}, index=[
    "split_replay", "Train_regression", "Test_transformation", "schema_equality",
    "Test_integrity", "feature_scope", "target_and_drift_scope",
    "prohibited_methods", "Reviewer_handoff",
])
display(final_stage_3_3_status)
print("Stage 3.3 implementation complete — pending final Reviewer validation.")


,result
split_replay,PASS — locked split and both stable ID hashes ...
Train_regression,PASS — Phase A Train accepted results unchanged
Test_transformation,PASS — frozen rules applied identically to Hol...
schema_equality,PASS — Train/Test ordered 29-column schemas id...
Test_integrity,"PASS — Test ID, rows, raw fields, and sealed l..."
feature_scope,PASS — no new feature or feature redesign
target_and_drift_scope,PASS — no Test target analysis or Train/Test d...
prohibited_methods,"PASS — no WOE, IV, binning, imbalance handling..."
Reviewer_handoff,PENDING — independent final Reviewer validatio...


Stage 3.3 implementation complete — pending final Reviewer validation.
